In [ ]:
#PPO Hyperparameter tuning
from Singe_ag_Environment_SparseR import SingleSatelliteEnvSR
from gymnasium.utils.env_checker import check_env
import traceback
print(type(SingleSatelliteEnvSR))
# This will catch many common issues
env = SingleSatelliteEnvSR("None")
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print("Environment has issues:")
    traceback.print_exc()

In [ ]:
import optuna

from cleanrl_utils.tuner import Tuner

tuner = Tuner(
    script="cleanrl/ppo.py",
    metric="charts/episodic_return",
    metric_last_n_average_window=50,
    direction="maximize",
    aggregation_type="average",
    target_scores={
        "SingleSatelliteEnvSR":None
    },
    params_fn=lambda trial: {
        "learning_rate": trial.suggest_float("learning_rate", 0.0003, 0.003, log=True),
        "num_minibatches": trial.suggest_categorical("num_minibatches", [1, 2, 4]),
        "update_epochs": trial.suggest_categorical("update_epochs", [1, 2, 4, 8]),
        "num_steps": trial.suggest_categorical("num_steps", [5, 16, 32, 64, 128]),
        "vf_coef": trial.suggest_float("vf_coef", 0, 1),
        "max_grad_norm": trial.suggest_float("max_grad_norm", 0, 1),
        "gae_lambda":trial.suggest_float("gae_lambda",0,1),
        "ent_coef":trial.suggest_float("ent_coef",0,1),
        "clip_coef":trial.suggest_float("clip_coef",0,1),
        "total_timesteps": 500000,
        "num_envs": 256,
    },
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    sampler=optuna.samplers.TPESampler(),
)
tuner.tune(
    num_trials=1,
    num_seeds=5,
)